# **Object-Oriented Programming (OOP) Exercise**  
**Objective:** Apply **Encapsulation, Inheritance, Polymorphism, and Method Overriding** in a practical exercise.  

---

## **Scenario: Library Management System**
You are tasked with building a **Library Management System** using Object-Oriented Programming in Python. The system should manage **Books** and **Users** with basic operations.

---

## **Exercise Instructions**

### **1️⃣ Create a `Book` Class**  
✅ Attributes:  
- `title` (string)  
- `author` (string)  
- `isbn` (string)  
- `available` (boolean, defaults to `True`)  

✅ Methods:  
- `__str__()`: Returns book details in a readable format.  
- `borrow()`: Marks the book as unavailable (`False`).  
- `return_book()`: Marks the book as available (`True`).  

**Example Usage**
```python
book1 = Book("Python Programming", "John Doe", "978-1234567890")
print(book1)  # Output: "Python Programming by John Doe (ISBN: 978-1234567890)"
book1.borrow()
print(book1.available)  # Output: False
book1.return_book()
print(book1.available)  # Output: True
```

---

### **2️⃣ Create a `User` Class**  
✅ Attributes:  
- `name` (string)  
- `user_id` (integer)  
- `borrowed_books` (list of books, empty by default)  

✅ Methods:  
- `borrow_book(book)`: Adds a book to `borrowed_books` if available.  
- `return_book(book)`: Removes a book from `borrowed_books`.  

**Example Usage**
```python
user1 = User("Alice", 101)
user1.borrow_book(book1)  # Alice borrows "Python Programming"
print(user1.borrowed_books)  # Output: ["Python Programming"]
user1.return_book(book1)
print(user1.borrowed_books)  # Output: []
```

---

### **3️⃣ Create a `Library` Class**
✅ Attributes:  
- `books` (list of all books in the library)  
- `users` (list of all users in the system)  

✅ Methods:  
- `add_book(book)`: Adds a new book to the library.  
- `add_user(user)`: Registers a new user.  
- `list_available_books()`: Prints all available books.  
- `borrow_book(user, book_title)`: Allows a user to borrow a book if available.  
- `return_book(user, book_title)`: Allows a user to return a book.  

**Example Usage**
```python
library = Library()
library.add_book(book1)
library.add_user(user1)
library.list_available_books()  # Should show "Python Programming"
library.borrow_book(user1, "Python Programming")
library.list_available_books()  # Should now be empty
library.return_book(user1, "Python Programming")
library.list_available_books()  # Should show the book again
```

---

### **4️⃣ Implement Inheritance**
✅ Create a subclass **`PremiumUser`** that inherits from `User`:  
- Allows borrowing **more than 3 books at a time**.  

**Example**
```python
premium_user = PremiumUser("Bob", 102)
premium_user.borrow_book(book1)  # Can borrow more than 3 books
```

---

### **5️⃣ Implement Polymorphism**
✅ Modify the `__str__()` method in `Book`, `User`, and `Library` to return useful information when printed.

---

## **Bonus Challenge **
🔹 Implement **a search function** to find books by title.  
🔹 Store book data in a JSON file and load it when the system starts.  

---

In [2]:
import json
from pathlib import Path


class Book:
    def __init__(self, title: str, author: str, isbn: str):
        self.title = title
        self.author = author
        self.isbn = isbn
        self.available = True

    def __str__(self) -> str:
        status = "available" if self.available else "borrowed"
        return f"{self.title} by {self.author} (ISBN: {self.isbn}) [{status}]"

    def borrow(self) -> None:
        self.available = False

    def return_book(self) -> None:
        self.available = True


class User:
    MAX_BOOKS = 3

    def __init__(self, name: str, user_id: int):
        self.name = name
        self.user_id = user_id
        self.borrowed_books: list[Book] = []

    def __str__(self) -> str:
        titles = ", ".join(b.title for b in self.borrowed_books) or "none"
        return f"User {self.name} (ID: {self.user_id}) — borrowed: {titles}"

    def borrow_book(self, book: Book) -> bool:
        if not book.available:
            print(f"'{book.title}' is not available.")
            return False
        if len(self.borrowed_books) >= self.MAX_BOOKS:
            print(f"{self.name} has reached the borrow limit ({self.MAX_BOOKS}).")
            return False
        book.borrow()
        self.borrowed_books.append(book)
        return True

    def return_book(self, book: Book) -> bool:
        if book not in self.borrowed_books:
            print(f"{self.name} did not borrow '{book.title}'.")
            return False
        self.borrowed_books.remove(book)
        book.return_book()
        return True


class PremiumUser(User):
    MAX_BOOKS = 10

    def __str__(self) -> str:
        return f"Premium {super().__str__()}"


class Library:
    def __init__(self, data_file: str | None = None):
        self.books: list[Book] = []
        self.users: list[User] = []
        if data_file and Path(data_file).exists():
            self.load_from_json(data_file)

    def __str__(self) -> str:
        return f"Library — {len(self.books)} books, {len(self.users)} users"

    def add_book(self, book: Book) -> None:
        self.books.append(book)

    def add_user(self, user: User) -> None:
        self.users.append(user)

    def list_available_books(self) -> None:
        available = [b for b in self.books if b.available]
        if not available:
            print("No books available.")
            return
        for book in available:
            print(f"  - {book.title}")

    def _find_book(self, title: str) -> Book | None:
        for book in self.books:
            if book.title == title:
                return book
        return None

    def search_by_title(self, keyword: str) -> list[Book]:
        keyword = keyword.lower()
        return [b for b in self.books if keyword in b.title.lower()]

    def borrow_book(self, user: User, book_title: str) -> bool:
        book = self._find_book(book_title)
        if book is None:
            print(f"Book '{book_title}' not found.")
            return False
        return user.borrow_book(book)

    def return_book(self, user: User, book_title: str) -> bool:
        book = self._find_book(book_title)
        if book is None:
            print(f"Book '{book_title}' not found.")
            return False
        return user.return_book(book)

    def save_to_json(self, path: str) -> None:
        data = [
            {"title": b.title, "author": b.author, "isbn": b.isbn, "available": b.available}
            for b in self.books
        ]
        Path(path).write_text(json.dumps(data, indent=2))

    def load_from_json(self, path: str) -> None:
        raw = json.loads(Path(path).read_text())
        for entry in raw:
            book = Book(entry["title"], entry["author"], entry["isbn"])
            book.available = entry.get("available", True)
            self.books.append(book)


# ── Demo ──────────────────────────────────────────────────────────────────────

book1 = Book("Python Programming", "John Doe", "978-1234567890")
book2 = Book("FastAPI Guide", "Jane Smith", "978-0987654321")
book3 = Book("DevOps Handbook", "Ali Mokh", "978-1111111111")
book4 = Book("Docker Deep Dive", "Bob Lee", "978-2222222222")

print(book1)
book1.borrow()
print("Available after borrow:", book1.available)
book1.return_book()
print("Available after return:", book1.available)
print()

user1 = User("Alice", 101)
user1.borrow_book(book1)
print("Alice borrowed:", [b.title for b in user1.borrowed_books])
user1.return_book(book1)
print("Alice borrowed after return:", user1.borrowed_books)
print()

library = Library()
library.add_book(book1)
library.add_book(book2)
library.add_user(user1)

print(library)
print("Available books:")
library.list_available_books()

library.borrow_book(user1, "Python Programming")
print("\nAfter borrow:")
library.list_available_books()

library.return_book(user1, "Python Programming")
print("\nAfter return:")
library.list_available_books()
print()

premium_user = PremiumUser("Bob", 102)
library.add_user(premium_user)
for b in [book2, book3, book4]:
    if b not in library.books:
        library.add_book(b)
    premium_user.borrow_book(b)

print(premium_user)
print(f"Bob borrowed {len(premium_user.borrowed_books)} books (PremiumUser limit > 3)")
print()

print("Search 'python':", [b.title for b in library.search_by_title("python")])

library.save_to_json("library_books.json")
library2 = Library("library_books.json")
print(f"\nLoaded from JSON: {library2}")
print("Books loaded:", [b.title for b in library2.books])

Python Programming by John Doe (ISBN: 978-1234567890) [available]
Available after borrow: False
Available after return: True

Alice borrowed: ['Python Programming']
Alice borrowed after return: []

Library — 2 books, 1 users
Available books:
  - Python Programming
  - FastAPI Guide

After borrow:
  - FastAPI Guide

After return:
  - Python Programming
  - FastAPI Guide

Premium User Bob (ID: 102) — borrowed: FastAPI Guide, DevOps Handbook, Docker Deep Dive
Bob borrowed 3 books (PremiumUser limit > 3)

Search 'python': ['Python Programming']

Loaded from JSON: Library — 4 books, 0 users
Books loaded: ['Python Programming', 'FastAPI Guide', 'DevOps Handbook', 'Docker Deep Dive']
